This is the initial data loading script for drivers. It is not really streaming though

In [0]:
%python
%pip install faker

catalog = "workspace"
schema = dbName = db = "streaming_cdc_test"

spark.sql(f'USE CATALOG `{catalog}`') 
spark.sql(f'USE SCHEMA `{schema}`')

spark.sql(f'CREATE VOLUME IF NOT EXISTS `{catalog}`.`{db}`.`raw_data`')
volume_folder =  f"/Volumes/{catalog}/{db}/raw_data"


print(f"generating the driver data under {volume_folder}...")
from pyspark.sql import functions as F
from faker import Faker
from collections import OrderedDict
import uuid
fake = Faker()
import random
import time
from pyspark.sql.types import StringType, IntegerType

fake_name = F.udf(lambda _: fake.name(), StringType())
fake_car_number = F.udf(lambda _: fake.license_plate(), StringType())
fake_experience = F.udf(lambda _: fake.random_int(min=1, max=30), IntegerType())
fake_rating = F.udf(lambda _: fake.random_int(min=1, max=5), IntegerType())

print("New iteration started, generating data...")
df = spark.range(10000).repartition(10)

df = df.withColumn('id', F.expr("cast(cast(rand()*10000+1 as int) as string)")) #this might create duplicates, that will be filtered in the silver layer

df = df.withColumn("name", fake_name(F.col('id')))
df = df.withColumn("car_number", fake_car_number(F.col('id')))
df = df.withColumn("experience", fake_experience(F.col('id')))
df_drivers = df.withColumn("rating", fake_rating(F.col('id')))
df_drivers.repartition(10).write.format("json").mode("overwrite").save(volume_folder+'/drivers')
print("----PREVIEW----")
df_drivers.show(5)



Now for rides table

In [0]:
%python
%pip install faker

catalog = "workspace"
schema = dbName = db = "streaming_cdc_test"

spark.sql(f'USE CATALOG `{catalog}`') 
spark.sql(f'USE SCHEMA `{schema}`')

spark.sql(f'CREATE VOLUME IF NOT EXISTS `{catalog}`.`{db}`.`raw_data`')
volume_folder =  f"/Volumes/{catalog}/{db}/raw_data"


print(f"generating the rides data under {volume_folder}...")
from pyspark.sql import functions as F
from faker import Faker
from collections import OrderedDict
import uuid
fake = Faker()
import random
import time
from pyspark.sql.types import StringType, IntegerType

fake_distance = F.udf(lambda _: fake.random_int(min=1, max=50), IntegerType())
fake_cost = F.udf(lambda _: fake.random_int(min=1, max=200), IntegerType())

print("New iteration started, generating data...")
df = spark.range(20000).repartition(20)

df = df.withColumn('driver_id', F.expr("cast(cast(rand()*10000+1 as int) as string)")) #this might create duplicates, that will be filtered in the silver layer
df = df.withColumn('ride_id', F.expr("cast(cast(rand()*20000+1 as int) as string)"))

df = df.withColumn("distance", fake_distance(F.col('id')))
df_rides = df.withColumn("cost", fake_cost(F.col('id'))).drop('id')
df_rides.repartition(20).write.format("json").mode("overwrite").save(volume_folder+'/rides')
print("----PREVIEW----")
df_rides.show(5)


The following is the literal streaming loop

In [0]:
%python
%pip install faker

catalog = "workspace"
schema = dbName = db = "streaming_cdc_test"

spark.sql(f'USE CATALOG `{catalog}`')
spark.sql(f'USE SCHEMA `{schema}`')

spark.sql(f'CREATE VOLUME IF NOT EXISTS `{catalog}`.`{db}`.`raw_data`')
volume_folder =  f"/Volumes/{catalog}/{db}/raw_data"

print(f"generating the data under {volume_folder}...")
from pyspark.sql import functions as F
from faker import Faker
from collections import OrderedDict
import uuid
fake = Faker()
import random
import time
from pyspark.sql.types import StringType, IntegerType

fake_id = F.udf(lambda _: random.choice(drivers_range), StringType())
fake_name = F.udf(lambda _: fake.name(), StringType())
fake_car_number = F.udf(lambda _: fake.license_plate(), StringType())
fake_experience = F.udf(lambda _: fake.random_int(min=1, max=30), IntegerType())
fake_rating = F.udf(lambda _: fake.random_int(min=1, max=5), IntegerType())

while(True):
    try:
        print("New iteration started, generating data...")
        df = spark.range(1000).repartition(10)
        df = df.withColumn("id", fake_id(F.col('id')))
        df = df.withColumn("name", fake_name(F.col('id')))
        df = df.withColumn("car_number", fake_car_number(F.col('id')))
        df = df.withColumn("experience", fake_experience(F.col('id')))
        df_drivers = df.withColumn("rating", fake_rating(F.col('id')))
        df_drivers.repartition(10).write.format("json").mode("append").save(volume_folder+'/drivers')
        print("----PREVIEW----")
        df_drivers.show()

        
        time.sleep(30)
    except Exception as e:
        print(e)
        continue
    
